In [104]:
import os
import copy
import json
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [105]:


DATA_ROOT = Path("./prepared_balanced_nilm")
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("./prepared_balanced_nilm")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

APPLIANCE = "washing_machine"
TARGET_BUILDING = "building_04"

TRAIN_BUILDINGS = ["building_01", "building_02"]
VAL_BUILDING = "building_03"

WINDOW_SIZE = 129
BATCH_SIZE = 256
EVAL_BATCH_SIZE = 2048
NUM_WORKERS = 0

EPOCHS = 8
LR = 5e-5

ACTIVITY_THRESHOLD = 2.0
CALIBRATION_RATIO = 0.05
CALIBRATION_MODE = "all"
PROB_THRESHOLD = 0.45
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [106]:
def building_dir(building):
    return DATA_ROOT / building

def csv_path(building):
    return building_dir(building) / "prepared_timeseries.csv"

def metadata_path(building):
    return building_dir(building) / "metadata.json"

def center_path(building, appliance, kind):
    return building_dir(building) / f"{appliance}_{kind}_centers.npy"

AGGREGATE_COLUMN = "aggregate_norm"

def target_norm_column(appliance):
    return f"{appliance}_norm"

def target_weight_column(appliance):
    return f"{appliance}_weight"

def load_building_dataframe(building):
    return pd.read_csv(csv_path(building))

def load_building_metadata(building):
    with open(metadata_path(building), "r", encoding="utf-8") as f:
        return json.load(f)

def get_appliance_metadata(building, appliance):
    meta = load_building_metadata(building)
    if appliance in meta and isinstance(meta[appliance], dict):
        return meta[appliance]
    return meta

def infer_columns(df, appliance):
    agg_col = AGGREGATE_COLUMN
    tgt_col = target_norm_column(appliance)
    if agg_col not in df.columns:
        raise ValueError(f"Missing aggregate column: {agg_col}")
    if tgt_col not in df.columns:
        raise ValueError(f"Missing target column: {tgt_col}")
    return agg_col, tgt_col

In [107]:
def inverse_target_transform_torch(y_norm, meta):
    mean_ = float(meta.get("target_mean", meta.get("mean", 0.0)))
    std_ = float(meta.get("target_std", meta.get("std", 1.0)))
    use_log = bool(meta.get("target_log1p", meta.get("use_log", False)))
    y_t = y_norm * std_ + mean_
    if use_log:
        y = torch.expm1(y_t)
    else:
        y = y_t
    return torch.clamp(y, min=0.0)

In [ ]:
class MultiTaskWindowDataset(Dataset):
    def __init__(self, aggregate, target, centers, window_size=129, sample_weights=None, activity_threshold=2.0):
        self.aggregate = aggregate.astype(np.float32)
        self.target = target.astype(np.float32)
        self.centers = centers.astype(np.int64)
        self.window_size = window_size
        self.half = window_size // 2
        self.sample_weights = np.ones(len(self.target), dtype=np.float32) if sample_weights is None else sample_weights.astype(np.float32)
        self.activity_threshold = activity_threshold

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        c = int(self.centers[idx])
        l = c - self.half
        r = c + self.half + 1
        x = self.aggregate[l:r]
        y = self.target[c]
        w = self.sample_weights[c]
        label = 1.0 if y >= self.activity_threshold else 0.0
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)
        y = torch.tensor(y, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)
        w = torch.tensor(w, dtype=torch.float32)
        return x, y, label, w

In [109]:
def make_sample_weights(target_array, power_weight=1.0):
    t = np.abs(target_array.astype(np.float32))
    return 1.0 + power_weight * t

def prepare_building_arrays(building, appliance):
    df = load_building_dataframe(building)
    agg_col, tgt_col = infer_columns(df, appliance)
    aggregate = df[agg_col].to_numpy(dtype=np.float32)
    target = df[tgt_col].to_numpy(dtype=np.float32)
    if target_weight_column(appliance) in df.columns:
        sample_weights = df[target_weight_column(appliance)].to_numpy(dtype=np.float32)
    else:
        sample_weights = make_sample_weights(target)
    balanced_centers = np.load(center_path(building, appliance, "balanced")).astype(np.int64)
    all_centers = np.load(center_path(building, appliance, "all")).astype(np.int64)
    inactive_centers = np.load(center_path(building, appliance, "inactive")).astype(np.int64)

    active_path = center_path(building, appliance, "active")
    if active_path.exists():
        active_centers = np.load(active_path).astype(np.int64)
    else:
        active_centers = np.array([c for c in all_centers if c not in set(inactive_centers.tolist())], dtype=np.int64)

    return {
        "aggregate": aggregate,
        "target": target,
        "weights": sample_weights,
        "balanced_centers": balanced_centers,
        "all_centers": all_centers,
        "inactive_centers": inactive_centers,
        "active_centers": active_centers
    }

In [110]:
def make_train_loaders(appliance):
    train_sets = []
    for b in TRAIN_BUILDINGS:
        data = prepare_building_arrays(b, appliance)
        ds = MultiTaskWindowDataset(
            data["aggregate"], data["target"], data["balanced_centers"],
            window_size=WINDOW_SIZE, sample_weights=data["weights"],
            activity_threshold=ACTIVITY_THRESHOLD
        )
        train_sets.append(ds)

    train_ds = ConcatDataset(train_sets)

    val_data = prepare_building_arrays(VAL_BUILDING, appliance)
    val_ds = MultiTaskWindowDataset(
        val_data["aggregate"], val_data["target"], val_data["balanced_centers"],
        window_size=WINDOW_SIZE, sample_weights=val_data["weights"],
        activity_threshold=ACTIVITY_THRESHOLD
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader

def make_target_loaders(appliance, target_building, calibration_ratio=0.05, mode="all"):
    data = prepare_building_arrays(target_building, appliance)
    aggregate = data["aggregate"]
    target = data["target"]
    weights = data["weights"]
    balanced = data["balanced_centers"].copy()
    all_centers = data["all_centers"].copy()
    inactive = data["inactive_centers"].copy()

    rng = np.random.default_rng(SEED)

    if mode == "all":
        pool = all_centers.copy()
        rng.shuffle(pool)
        n_cal = max(1, int(len(pool) * calibration_ratio))
        cal_centers = np.sort(pool[:n_cal])
    else:
        total_n = max(1, int(len(all_centers) * calibration_ratio))
        n_bal = int(total_n * 0.5)
        n_inactive = total_n - n_bal
        bal_pool = balanced.copy()
        inact_pool = inactive.copy()
        rng.shuffle(bal_pool)
        rng.shuffle(inact_pool)
        sel = []
        if len(bal_pool) > 0 and n_bal > 0:
            sel.append(bal_pool[:min(n_bal, len(bal_pool))])
        if len(inact_pool) > 0 and n_inactive > 0:
            sel.append(inact_pool[:min(n_inactive, len(inact_pool))])
        if len(sel) == 0:
            pool = all_centers.copy()
            rng.shuffle(pool)
            n_cal = max(1, int(len(pool) * calibration_ratio))
            cal_centers = np.sort(pool[:n_cal])
        else:
            cal_centers = np.unique(np.concatenate(sel)).astype(np.int64)
            cal_centers = np.sort(cal_centers)

    cal_set = set(cal_centers.tolist())
    full_holdout_centers = np.array([c for c in all_centers if int(c) not in cal_set], dtype=np.int64)
    if len(full_holdout_centers) == 0 and len(all_centers) > 0:
        full_holdout_centers = all_centers[:1]

    cal_ds = MultiTaskWindowDataset(aggregate, target, cal_centers, window_size=WINDOW_SIZE, sample_weights=weights, activity_threshold=ACTIVITY_THRESHOLD)
    full_ds = MultiTaskWindowDataset(aggregate, target, full_holdout_centers, window_size=WINDOW_SIZE, sample_weights=weights, activity_threshold=ACTIVITY_THRESHOLD)

    cal_loader = DataLoader(cal_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    full_loader = DataLoader(full_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    return cal_loader, full_loader

In [111]:
class MultiTaskNILM(nn.Module):
    def __init__(self, input_dim=1, channels=64, dropout=0.1):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(input_dim, 32, 5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, channels, 5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.cls_head = nn.Linear(64, 1)
        self.reg_head = nn.Linear(64, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        feat = self.backbone(x)
        cls_logit = self.cls_head(feat).squeeze(-1)
        reg_out = self.reg_head(feat).squeeze(-1)
        return cls_logit, reg_out

In [112]:
class WeightedAsymmetricHuberLoss(nn.Module):
    def __init__(self, delta=1.0, underpredict_weight=2.0):
        super().__init__()
        self.delta = delta
        self.underpredict_weight = underpredict_weight

    def forward(self, pred, target, sample_weight=None):
        err = pred - target
        abs_err = torch.abs(err)
        huber = torch.where(abs_err <= self.delta, 0.5 * err ** 2, self.delta * (abs_err - 0.5 * self.delta))
        asym = torch.where(pred < target, self.underpredict_weight, 1.0)
        loss = huber * asym
        if sample_weight is not None:
            loss = loss * sample_weight
        return loss.mean()

class WeightedBCEWithLogitsLoss(nn.Module):
    def forward(self, logits, labels, sample_weight=None):
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        if sample_weight is not None:
            loss = loss * sample_weight
        return loss.mean()

def train_one_epoch_multitask(model, loader, optimizer, cls_loss_fn, reg_loss_fn, device, lambda_cls=1.0, lambda_reg=1.0):
    model.train()
    losses = []
    for x, y, label, w in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        w = w.to(device, non_blocking=True)

        optimizer.zero_grad()
        cls_logit, reg_out = model(x)

        cls_loss = cls_loss_fn(cls_logit, label, w)
        reg_loss = reg_loss_fn(reg_out, y, w)
        loss = lambda_cls * cls_loss + lambda_reg * reg_loss

        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return float(np.mean(losses)) if len(losses) else 0.0

In [ ]:
def collect_outputs_multitask(model, loader, device, meta, prob_threshold=0.45):
    model.eval()
    preds, trues, probs = [], [], []

    with torch.inference_mode():
        for x, y, label, w in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            cls_logit, reg_out = model(x)
            prob = torch.sigmoid(cls_logit)

            prob_np = prob.cpu().numpy()
            reg_np = inverse_target_transform_torch(reg_out, meta).cpu().numpy()
            true_np = inverse_target_transform_torch(y, meta).cpu().numpy()

            reg_np[prob_np < prob_threshold] = 0.0

            preds.append(reg_np)
            trues.append(true_np)
            probs.append(prob_np)

    if len(preds) == 0:
        return None, None, None

    return np.concatenate(preds), np.concatenate(trues), np.concatenate(probs)

In [113]:
def evaluate_multitask(model, loader, device, meta, prob_threshold=0.45):
    model.eval()
    preds, trues, probs = [], [], []

    with torch.inference_mode():
        for x, y, label, w in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            cls_logit, reg_out = model(x)
            prob = torch.sigmoid(cls_logit)

            prob_np = prob.cpu().numpy()
            reg_np = inverse_target_transform_torch(reg_out, meta).cpu().numpy()
            true_np = inverse_target_transform_torch(y, meta).cpu().numpy()

            reg_np[prob_np < prob_threshold] = 0.0

            preds.append(reg_np)
            trues.append(true_np)
            probs.append(prob_np)

    if len(preds) == 0:
        return None

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    probs = np.concatenate(probs)

    mae = float(np.mean(np.abs(trues - preds)))
    ss_res = float(np.sum((trues - preds) ** 2))
    ss_tot = float(np.sum((trues - np.mean(trues)) ** 2))
    r2 = 0.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot
    rel_mae_pct = float(100.0 * mae / max(np.mean(trues), 1e-8))

    true_on = (trues >= ACTIVITY_THRESHOLD).astype(np.int32)
    pred_on = (probs >= prob_threshold).astype(np.int32)

    tp = int(np.sum((true_on == 1) & (pred_on == 1)))
    tn = int(np.sum((true_on == 0) & (pred_on == 0)))
    fp = int(np.sum((true_on == 0) & (pred_on == 1)))
    fn = int(np.sum((true_on == 1) & (pred_on == 0)))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)

    return {
        "regression": {
            "mae": mae,
            "r2": r2,
            "rel_mae_pct": rel_mae_pct,
            "pred_mean": float(np.mean(preds)),
            "true_mean": float(np.mean(trues)),
            "num_samples": len(trues)
        },
        "binary": {
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1, "accuracy": acc
        }
    }

In [114]:
train_loader, val_loader = make_train_loaders(APPLIANCE)
calibration_loader, full_holdout_loader = make_target_loaders(
    APPLIANCE, TARGET_BUILDING, calibration_ratio=CALIBRATION_RATIO, mode=CALIBRATION_MODE
)

source_meta = get_appliance_metadata(VAL_BUILDING, APPLIANCE)
target_meta = get_appliance_metadata(TARGET_BUILDING, APPLIANCE)

model = MultiTaskNILM().to(DEVICE)
cls_loss_fn = WeightedBCEWithLogitsLoss()
reg_loss_fn = WeightedAsymmetricHuberLoss(delta=1.0, underpredict_weight=2.0)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    loss = train_one_epoch_multitask(
        model, train_loader, optimizer, cls_loss_fn, reg_loss_fn, DEVICE,
        lambda_cls=1.0, lambda_reg=1.0
    )
    print(f"[MT {epoch:02d}/{EPOCHS}] loss={loss:.4f} time={time.time()-t0:.1f}s")

for epoch in range(1, 4):
    t0 = time.time()
    loss = train_one_epoch_multitask(
        model, calibration_loader, optimizer, cls_loss_fn, reg_loss_fn, DEVICE,
        lambda_cls=0.3, lambda_reg=1.0
    )
    print(f"[CAL {epoch:02d}/3] loss={loss:.4f} time={time.time()-t0:.1f}s")

[MT 01/8] loss=10.3902 time=3.8s
[MT 02/8] loss=6.7734 time=3.8s
[MT 03/8] loss=4.0208 time=3.5s
[MT 04/8] loss=2.9174 time=3.5s
[MT 05/8] loss=2.5318 time=3.9s
[MT 06/8] loss=2.2764 time=3.6s
[MT 07/8] loss=2.0625 time=3.6s
[MT 08/8] loss=1.9190 time=3.5s
[CAL 01/3] loss=0.4590 time=34.3s
[CAL 02/3] loss=0.4155 time=34.6s
[CAL 03/3] loss=0.3971 time=33.2s


In [115]:
results = evaluate_multitask(
    model, full_holdout_loader, DEVICE, target_meta, prob_threshold=PROB_THRESHOLD
)

print("REGRESSION")
for k, v in results["regression"].items():
    print(f"{k}: {v}")

print("\nBINARY")
for k, v in results["binary"].items():
    print(f"{k}: {v}")

REGRESSION
mae: 0.2632487416267395
r2: -0.11840063796227351
rel_mae_pct: 110.80157470703125
pred_mean: 0.4351239800453186
true_mean: 0.23758575320243835
num_samples: 15527166

BINARY
tp: 865490
tn: 13741759
fp: 856617
fn: 63300
precision: 0.5025762046144635
recall: 0.931846811442845
f1: 0.6529789727778935
accuracy: 0.9407543527260545


In [116]:
def sweep_prob_thresholds(classifier, regressor, loader, device, meta,
                          thresholds=(0.3, 0.4, 0.5, 0.6, 0.7, 0.8),
                          activity_threshold=2.0):
    classifier.eval()
    regressor.eval()

    all_probs = []
    all_preds = []
    all_trues = []

    with torch.inference_mode():
        for x, y, label, w in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            prob = torch.sigmoid(classifier(x))
            reg = regressor(x)

            prob_np = prob.cpu().numpy()
            reg_np = inverse_target_transform_torch(reg, meta).cpu().numpy()
            true_np = inverse_target_transform_torch(y, meta).cpu().numpy()

            all_probs.append(prob_np)
            all_preds.append(reg_np)
            all_trues.append(true_np)

    if len(all_probs) == 0:
        return pd.DataFrame(columns=["threshold", "mae", "r2", "rel_mae_pct", "pred_mean", "true_mean", "precision", "recall", "f1", "accuracy"])

    probs = np.concatenate(all_probs)
    preds_raw = np.concatenate(all_preds)
    trues = np.concatenate(all_trues)

    rows = []
    for thr in thresholds:
        preds = preds_raw.copy()
        preds[probs < thr] = 0.0

        reg = stage_metrics(trues, preds)
        binm = binary_metrics(trues, probs, thr=thr, activity_threshold=activity_threshold)

        rows.append({
            "threshold": thr,
            "mae": reg["mae"],
            "r2": reg["r2"],
            "rel_mae_pct": reg["rel_mae_pct"],
            "pred_mean": reg["pred_mean"],
            "true_mean": reg["true_mean"],
            "precision": binm["precision"],
            "recall": binm["recall"],
            "f1": binm["f1"],
            "accuracy": binm["accuracy"]
        })

    return pd.DataFrame(rows)

In [117]:
sweep_df = sweep_prob_thresholds(
    classifier=classifier,
    regressor=regressor,
    loader=full_holdout_loader,
    device=DEVICE,
    meta=target_meta,
    thresholds=(0.3, 0.4, 0.5, 0.6, 0.7, 0.8),
    activity_threshold=ACTIVITY_THRESHOLD
)

print(sweep_df)

KeyboardInterrupt: 

In [ ]:
# cls_train_loader, reg_train_loader, cls_val_loader, calibration_loader, full_holdout_loader = build_classifier_and_regression_loaders(
#     APPLIANCE, TARGET_BUILDING, calibration_ratio=CALIBRATION_RATIO, activity_threshold=ACTIVITY_THRESHOLD
# )

# source_meta = get_appliance_metadata(VAL_BUILDING, APPLIANCE)
# target_meta = get_appliance_metadata(TARGET_BUILDING, APPLIANCE)

# classifier = ActivityClassifier().to(DEVICE)
# regressor = PowerRegressor().to(DEVICE)

# cls_loss_fn = WeightedBCEWithLogitsLoss()
# reg_loss_fn = WeightedAsymmetricHuberLoss(delta=1.0, underpredict_weight=2.0)

# opt_cls = AdamW(classifier.parameters(), lr=CLASSIFIER_LR, weight_decay=1e-4)
# opt_reg = AdamW(regressor.parameters(), lr=REGRESSOR_LR, weight_decay=1e-4)

# for epoch in range(1, CLASSIFIER_EPOCHS + 1):
#     t0 = time.time()
#     cls_loss = train_one_epoch_classifier(classifier, cls_train_loader, opt_cls, cls_loss_fn, DEVICE)
#     print(f"[CLS {epoch:02d}/{CLASSIFIER_EPOCHS}] loss={cls_loss:.4f} time={time.time()-t0:.1f}s")

# for epoch in range(1, REGRESSOR_EPOCHS + 1):
#     t0 = time.time()
#     reg_loss = train_one_epoch_regressor(regressor, reg_train_loader, opt_reg, reg_loss_fn, DEVICE)
#     print(f"[REG {epoch:02d}/{REGRESSOR_EPOCHS}] loss={reg_loss:.4f} time={time.time()-t0:.1f}s")

In [ ]:
preds, trues, probs = collect_two_stage_outputs(
    classifier, regressor, full_holdout_loader, DEVICE, target_meta, prob_threshold=PROB_THRESHOLD
)

if preds is not None:
    reg = stage_metrics(trues, preds)
    binm = binary_metrics(trues, probs, thr=PROB_THRESHOLD, activity_threshold=ACTIVITY_THRESHOLD)

    print("REGRESSION")
    for k, v in reg.items():
        print(f"{k}: {v}")

    print("\nBINARY")
    for k, v in binm.items():
        print(f"{k}: {v}")

KeyboardInterrupt: 

In [ ]:
best_threshold = 0.5

preds, trues, probs = collect_two_stage_outputs(
    classifier, regressor, full_holdout_loader, DEVICE, target_meta, prob_threshold=best_threshold
)

reg = stage_metrics(trues, preds)
binm = binary_metrics(trues, probs, thr=best_threshold, activity_threshold=ACTIVITY_THRESHOLD)

print("REGRESSION")
for k, v in reg.items():
    print(f"{k}: {v}")

print("\nBINARY")
for k, v in binm.items():
    print(f"{k}: {v}")

REGRESSION
mae: 0.2797147333621979
r2: -0.2314817963797986
rel_mae_pct: 117.73211669921875
pred_mean: 0.39622989296913147
true_mean: 0.23758575320243835

BINARY
tp: 705249
tn: 13848666
fp: 749710
fn: 223541
precision: 0.4847208752961424
recall: 0.7593201907858612
f1: 0.5917141444002704
accuracy: 0.9373194696314833
